# Notebook 1: ML Price Forecasting
## IEOR E4010 — AI for Operations Research and Financial Engineering
### Columbia University, Spring 2026

---

## Overview

In this notebook you will build a machine learning model to forecast **real-time electricity prices** in New York City (NYISO Zone J). Accurate price forecasts are the foundation of smart EV charging: if we know a price spike is coming in 2 hours, we charge vehicles now; if prices will drop overnight, we wait.

### What you will do
1. **Explore** the 2025 NYISO price and weather dataset
2. **Engineer features** from lag prices, rolling statistics, and DA price signals
3. **Train a baseline XGBoost model** (provided)
4. **Design custom features** to beat the baseline (your main task)
5. **Export your model** to `submission/ml_model.pkl`

### Grading
Your ML score is based on **Mean Absolute Error (MAE) in $/MWh** on a held-out test set (months 7–12). Lower is better. You beat the baseline if your custom model's MAE < baseline MAE.

### Key insight
The day-ahead price (`dam_lbmp_mwh`) is the strongest single predictor of the real-time price — it is set by ISO clearing the market 12–36 hours ahead. Weather features help capture demand shocks that cause RT prices to deviate from DA.

In [ ]:
# ── Cell 0: Setup and Data Loading ──────────────────────────────────────────
import subprocess, sys, os

# Install dependencies (silent on Colab, fast if already installed)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "xgboost>=2.0", "scikit-learn>=1.3", "pandas>=2.0",
    "numpy>=1.24", "matplotlib>=3.7", "seaborn>=0.12"])

print("Dependencies installed.")

In [ ]:
import os, shutil

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

DATA_FILES = [
    "nyiso_prices_weather_nyc_2025.csv",
    "nyiso_rt_lbmp_nyc_2025_5min.csv",
    "nyc_fleet_profiles.json",
]

# Search order: data/, ../, ../../
search_parents = ["", "..", "../.."]

for filename in DATA_FILES:
    local_path = os.path.join(DATA_DIR, filename)
    if os.path.exists(local_path):
        print(f"  Found: {local_path}")
        continue
    copied = False
    for parent in search_parents:
        candidate = os.path.join(parent, filename) if parent else filename
        if os.path.exists(candidate):
            shutil.copy(candidate, local_path)
            print(f"  Copied {filename} from {parent or 'cwd'}")
            copied = True
            break
    if not copied:
        print(f"  WARNING: {filename} not found — place it in data/ or download from repo.")

print("\nData directory contents:", os.listdir(DATA_DIR))

---
## Section 1: Data Exploration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

# ── Load hourly data ─────────────────────────────────────────────────────────
df = pd.read_csv("data/nyiso_prices_weather_nyc_2025.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nBasic statistics for key price columns:")
print(df[["dam_lbmp_mwh", "rt_lbmp_mwh", "da_rt_spread_mwh"]].describe())

In [ ]:
# ── Plot 1: RT price over 2025 with seasonal color ───────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

season_colors = {1: "royalblue", 2: "royalblue", 3: "seagreen",
                 4: "seagreen", 5: "seagreen", 6: "tomato",
                 7: "tomato", 8: "tomato", 9: "orange",
                 10: "orange", 11: "royalblue", 12: "royalblue"}
season_names = {"royalblue": "Winter", "seagreen": "Spring/Fall",
                "tomato": "Summer", "orange": "Autumn"}

ax = axes[0]
for month, grp in df.groupby(df["timestamp"].dt.month):
    ax.plot(grp["timestamp"], grp["rt_lbmp_mwh"],
            color=season_colors[month], linewidth=0.5, alpha=0.7)
ax.set_title("NYC Real-Time LBMP — 2025 (hourly average)", fontsize=13)
ax.set_ylabel("Price ($/MWh)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

# Legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color=c, linewidth=2, label=l)
                   for c, l in season_names.items()]
ax.legend(handles=legend_elements, loc="upper left")

# Plot 2: 7-day rolling average smoothed
ax2 = axes[1]
df_roll = df.set_index("timestamp")["rt_lbmp_mwh"].rolling(168).mean()
ax2.fill_between(df_roll.index, df_roll, alpha=0.4, color="steelblue")
ax2.plot(df_roll.index, df_roll, color="steelblue", linewidth=1.5)
ax2.set_title("7-day rolling mean RT price", fontsize=13)
ax2.set_ylabel("Price ($/MWh)")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

plt.tight_layout()
plt.savefig("data/plot1_rt_prices.png", dpi=120, bbox_inches="tight")
plt.show()
print("Summer prices are consistently higher due to AC demand.")

In [ ]:
# ── Plot 2: Average hourly price profile (weekday vs weekend) ─────────────────
df["hour"] = df["timestamp"].dt.hour
df["is_weekend"] = (df["timestamp"].dt.dayofweek >= 5).astype(int)

hourly_avg = df.groupby(["hour", "is_weekend"])["rt_lbmp_mwh"].agg(["mean", "std"]).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
for is_we, label, color in [(0, "Weekday", "steelblue"), (1, "Weekend", "tomato")]:
    sub = hourly_avg[hourly_avg["is_weekend"] == is_we]
    ax.plot(sub["hour"], sub["mean"], label=label, color=color, linewidth=2)
    ax.fill_between(sub["hour"], sub["mean"] - sub["std"], sub["mean"] + sub["std"],
                    alpha=0.2, color=color)

ax.axvspan(16, 20, alpha=0.1, color="red", label="Evening peak (4–8 PM)")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Avg RT Price ($/MWh)")
ax.set_title("Average Hourly RT Price Profile — Weekday vs Weekend")
ax.legend()
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.savefig("data/plot2_hourly_profile.png", dpi=120, bbox_inches="tight")
plt.show()
print("Weekdays show a clear evening peak (4–8 PM). Weekends are flatter.")

In [ ]:
# ── Plot 3: DA vs RT scatter, colored by season ───────────────────────────────
df["season"] = df["timestamp"].dt.month.map(
    {12: "Winter", 1: "Winter", 2: "Winter",
     3: "Spring", 4: "Spring", 5: "Spring",
     6: "Summer", 7: "Summer", 8: "Summer",
     9: "Autumn", 10: "Autumn", 11: "Autumn"}
)

fig, ax = plt.subplots(figsize=(8, 7))
palette = {"Winter": "royalblue", "Spring": "seagreen",
           "Summer": "tomato", "Autumn": "orange"}

for season, grp in df.groupby("season"):
    ax.scatter(grp["dam_lbmp_mwh"], grp["rt_lbmp_mwh"],
               alpha=0.15, s=4, color=palette[season], label=season)

lim = max(df["dam_lbmp_mwh"].max(), df["rt_lbmp_mwh"].max()) * 1.05
ax.plot([0, lim], [0, lim], "k--", linewidth=1, alpha=0.5, label="DA = RT")
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel("Day-Ahead Price ($/MWh)")
ax.set_ylabel("Real-Time Price ($/MWh)")
ax.set_title("DA vs RT Prices by Season")
ax.legend(markerscale=5)
plt.tight_layout()
plt.savefig("data/plot3_da_rt_scatter.png", dpi=120, bbox_inches="tight")
plt.show()
corr = df[["dam_lbmp_mwh", "rt_lbmp_mwh"]].corr().iloc[0, 1]
print(f"DA-RT correlation: {corr:.3f}. Strong — DA price is the best single predictor.")

In [ ]:
# ── Plot 4: DA-RT Spread distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.hist(df["da_rt_spread_mwh"].clip(-100, 100), bins=100,
        color="steelblue", edgecolor="white", linewidth=0.3)
ax.axvline(0, color="red", linestyle="--", linewidth=1.5, label="DA = RT")
ax.set_xlabel("DA - RT Spread ($/MWh)")
ax.set_ylabel("Count (hours)")
ax.set_title("Distribution of DA-RT Price Spread")
ax.legend()

ax2 = axes[1]
spike_threshold = 20
spike_pct = (df["da_rt_spread_mwh"].abs() > spike_threshold).mean() * 100
df["is_spike"] = (df["da_rt_spread_mwh"].abs() > spike_threshold).astype(int)
df.groupby(["hour", "is_spike"]).size().unstack(fill_value=0).plot(
    kind="bar", ax=ax2, color=["steelblue", "tomato"], legend=True
)
ax2.set_xlabel("Hour of Day")
ax2.set_ylabel("Count")
ax2.set_title(f"Price Spikes (|spread|>20 $/MWh) by Hour\n{spike_pct:.1f}% of hours are spikes")
ax2.legend(["Normal", "Spike"])
ax2.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig("data/plot4_spread.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Spike hours concentration: {df.groupby('hour')['is_spike'].mean().idxmax()} h")

---
## Section 2: Baseline Feature Engineering (PROVIDED)

The function below computes:
- **Lag features**: RT prices at t-1h, t-2h, t-3h, t-6h, t-12h, t-24h, t-48h, t-168h
- **Rolling statistics**: mean and std over past 6h, 24h, 168h
- **DA price**: The strongest predictor (already in data)
- **DA-RT spread lag**: Yesterday's spread at this hour (t-24h)
- **Calendar features**: hour, day-of-week, month, weekend flag (cyclical encodings already in data)

In [ ]:
def engineer_features_baseline(df):
    """Compute baseline lag and rolling features for XGBoost."""
    df = df.copy()
    
    # Lag RT prices
    for lag in [1, 2, 3, 6, 12, 24, 48, 168]:
        df[f"rt_lag_{lag}h"] = df["rt_lbmp_mwh"].shift(lag)
    
    # Rolling mean and std (past only — no leakage)
    for window in [6, 24, 168]:
        df[f"rt_roll_mean_{window}h"] = (
            df["rt_lbmp_mwh"].shift(1).rolling(window, min_periods=window//2).mean()
        )
        df[f"rt_roll_std_{window}h"] = (
            df["rt_lbmp_mwh"].shift(1).rolling(window, min_periods=window//2).std()
        )
    
    # DA price (strongest single predictor)
    # dam_lbmp_mwh is already in the dataframe — no shift needed
    # (DA is known 12-36h ahead; we can use it as a feature for forecasting RT)
    
    # DA-RT spread lag (yesterday at same hour)
    df["spread_lag_24h"] = df["da_rt_spread_mwh"].shift(24)
    
    # RT price std lag
    df["rt_std_lag_1h"] = df["rt_lbmp_std_mwh"].shift(1)
    
    return df


# Apply
df = engineer_features_baseline(df)

# Define baseline feature list
BASELINE_FEATURES = (
    [f"rt_lag_{lag}h" for lag in [1, 2, 3, 6, 12, 24, 48, 168]] +
    [f"rt_roll_mean_{w}h" for w in [6, 24, 168]] +
    [f"rt_roll_std_{w}h" for w in [6, 24, 168]] +
    ["dam_lbmp_mwh", "spread_lag_24h", "rt_std_lag_1h"] +
    ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]
)

TARGET = "rt_lbmp_mwh"

print(f"Baseline feature count: {len(BASELINE_FEATURES)}")
print("Features:", BASELINE_FEATURES)

---
## Section 3: Train/Test Split

**Chronological split**: train on months 1–8, validate on months 9–12.

**Why not random shuffle?** Electricity prices are strongly autocorrelated. A random split would leak future information into the training set (e.g., the model sees next week's prices when learning to predict this week). This would produce unrealistically optimistic validation metrics.

In [ ]:
# Drop rows with NaN features (first ~168 hours have incomplete rolling windows)
df_model = df[BASELINE_FEATURES + [TARGET, "timestamp", "da_rt_spread_mwh"]].dropna()

# Chronological 70/30 split by date
split_date = pd.Timestamp("2025-09-01")

train_mask = df_model["timestamp"] < split_date
val_mask   = df_model["timestamp"] >= split_date

X_train = df_model.loc[train_mask, BASELINE_FEATURES].values
y_train = df_model.loc[train_mask, TARGET].values
X_val   = df_model.loc[val_mask, BASELINE_FEATURES].values
y_val   = df_model.loc[val_mask, TARGET].values

print(f"Train: {X_train.shape[0]} rows  ({df_model.loc[train_mask,'timestamp'].min().date()} to {df_model.loc[train_mask,'timestamp'].max().date()})")
print(f"Val:   {X_val.shape[0]} rows  ({df_model.loc[val_mask,'timestamp'].min().date()} to {df_model.loc[val_mask,'timestamp'].max().date()})")

# ── Demo: why random shuffle is wrong ────────────────────────────────────────
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

X_all = df_model[BASELINE_FEATURES].values
y_all = df_model[TARGET].values

Xr_train, Xr_val, yr_train, yr_val = train_test_split(X_all, y_all, test_size=0.3, random_state=42)
m_leaky = xgb.XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
m_leaky.fit(Xr_train, yr_train)
mae_leaky = mean_absolute_error(yr_val, m_leaky.predict(Xr_val))

print(f"\nLeaky (random split) MAE: {mae_leaky:.2f} $/MWh  <-- suspiciously low!")
print("This is artificially low because future prices leak into training data.")

---
## Section 4: Baseline XGBoost Model (PROVIDED)

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_baseline = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method="hist",   # fast on CPU
    early_stopping_rounds=20,
    eval_metric="mae",
)

model_baseline.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

pred_baseline = model_baseline.predict(X_val)

mae_baseline  = mean_absolute_error(y_val, pred_baseline)
rmse_baseline = mean_squared_error(y_val, pred_baseline) ** 0.5
r2_baseline   = r2_score(y_val, pred_baseline)

print("=== Baseline XGBoost ====")
print(f"  MAE  : {mae_baseline:.3f} $/MWh")
print(f"  RMSE : {rmse_baseline:.3f} $/MWh")
print(f"  R²   : {r2_baseline:.4f}")

In [ ]:
# ── Feature importance ────────────────────────────────────────────────────────
feat_imp = pd.Series(
    model_baseline.feature_importances_,
    index=BASELINE_FEATURES
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.head(20).plot(kind="barh", ax=ax, color="steelblue")
ax.invert_yaxis()
ax.set_xlabel("Feature Importance (gain)")
ax.set_title("Top-20 Baseline Feature Importances")
plt.tight_layout()
plt.savefig("data/plot5_feat_importance_baseline.png", dpi=120, bbox_inches="tight")
plt.show()
print("dam_lbmp_mwh dominates — confirms DA price is the best single predictor.")

In [ ]:
# ── Evaluate on spike hours ───────────────────────────────────────────────────
val_df = df_model.loc[val_mask].copy()
val_df["pred_baseline"] = pred_baseline
val_df["error_baseline"] = (val_df["pred_baseline"] - val_df[TARGET]).abs()

spike_mask = val_df["da_rt_spread_mwh"].abs() > 20
normal_mask = ~spike_mask

print(f"Spike hours: {spike_mask.sum()} ({spike_mask.mean()*100:.1f}% of val)")
print(f"  Baseline MAE on spikes : {val_df.loc[spike_mask, 'error_baseline'].mean():.3f} $/MWh")
print(f"  Baseline MAE on normal : {val_df.loc[normal_mask, 'error_baseline'].mean():.3f} $/MWh")
print("\nSpike hours are much harder to forecast — your custom model should focus here.")

---
## Section 5: TODO — Student Feature Engineering

Your task is to implement `engineer_features_custom()`. The function should return a DataFrame with **additional columns** beyond the baseline.

**Suggestions:**
- **Weather features**: `cooling_degree_hours`, `heating_degree_hours`, `temp_anomaly`, `humidity_discomfort`, `solar_generation_proxy`, `wind_power_proxy`
- **Interaction terms**: `cooling_degree_hours × is_peak_hour` (AC load during peak)
- **Longer lags**: t-72h (3-day), t-120h (5-day)
- **DA×temperature**: captures demand-driven deviation from DA forecast

After defining the function, train a new XGBoost model and compare to the baseline.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# === TODO: Engineer Custom Features ===
# Add weather features, interaction terms, longer lags, Fourier terms
#
# Instructions:
#   1. Call engineer_features_baseline(df) inside your function to include the
#      baseline features.
#   2. Add at least 5 new columns.
#   3. Return the modified DataFrame.
# ──────────────────────────────────────────────────────────────────────────────

def engineer_features_custom(df):
    """
    === TODO ===
    Add custom features to improve the XGBoost model.

    Suggested features:
    - Weather: cooling_degree_hours, heating_degree_hours, temp_anomaly,
               humidity_discomfort, solar_generation_proxy, wind_power_proxy
    - Interactions: cooling_degree_hours × is_peak_hour
    - Longer lags: t-72h, t-120h
    - Interaction: dam_lbmp_mwh × cooling_degree_hours
    - Volatility: rt_lbmp_std_mwh lagged by 24h

    Returns: df with additional columns added
    """
    # === SUGGESTED SOLUTION ===
    df = engineer_features_baseline(df)

    # Weather load proxies
    df["cdh_feature"]               = df["cooling_degree_hours"]
    df["hdh_feature"]               = df["heating_degree_hours"]
    df["temp_anomaly_feature"]      = df["temp_anomaly"]
    df["humidity_discomfort_feat"]  = df["humidity_discomfort"]
    df["solar_proxy"]               = df["solar_generation_proxy"]
    df["wind_proxy"]                = df["wind_power_proxy"]

    # Peak-hour interaction (evenings 4–8 PM are when AC + cooking overlap)
    df["is_peak_hour"] = df["hour"].isin([16, 17, 18, 19, 20]).astype(int)
    df["hot_peak"]     = df["cdh_feature"] * df["is_peak_hour"]

    # Longer lags (multi-day periodicity)
    df["rt_lag_72h"]  = df["rt_lbmp_mwh"].shift(72)
    df["rt_lag_120h"] = df["rt_lbmp_mwh"].shift(120)

    # DA-price × temperature interaction
    # On hot days, actual demand exceeds DA forecast → RT > DA
    df["dam_times_cdh"] = df["dam_lbmp_mwh"] * df["cooling_degree_hours"]
    df["dam_times_hdh"] = df["dam_lbmp_mwh"] * df["heating_degree_hours"]

    # Intra-hour volatility lagged 24h (yesterday's variance signal)
    df["rt_std_lag_24h"] = df["rt_lbmp_std_mwh"].shift(24)

    # Month as cyclical feature
    month = df["timestamp"].dt.month if "timestamp" in df.columns else df["month"]
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)

    return df


print("engineer_features_custom() defined.")

In [ ]:
# Apply custom features
df_custom = engineer_features_custom(df)

CUSTOM_FEATURES = (
    BASELINE_FEATURES +
    ["cdh_feature", "hdh_feature", "temp_anomaly_feature",
     "humidity_discomfort_feat", "solar_proxy", "wind_proxy",
     "is_peak_hour", "hot_peak",
     "rt_lag_72h", "rt_lag_120h",
     "dam_times_cdh", "dam_times_hdh",
     "rt_std_lag_24h",
     "month_sin", "month_cos"]
)

df_custom_model = df_custom[CUSTOM_FEATURES + [TARGET, "timestamp", "da_rt_spread_mwh"]].dropna()

train_mask_c = df_custom_model["timestamp"] < split_date
val_mask_c   = df_custom_model["timestamp"] >= split_date

X_train_c = df_custom_model.loc[train_mask_c, CUSTOM_FEATURES].values
y_train_c = df_custom_model.loc[train_mask_c, TARGET].values
X_val_c   = df_custom_model.loc[val_mask_c, CUSTOM_FEATURES].values
y_val_c   = df_custom_model.loc[val_mask_c, TARGET].values

print(f"Custom feature count: {len(CUSTOM_FEATURES)}")
print(f"Train: {X_train_c.shape[0]} rows, Val: {X_val_c.shape[0]} rows")

In [ ]:
# Train custom XGBoost
model_custom = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.04,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    tree_method="hist",
    early_stopping_rounds=30,
    eval_metric="mae",
)

model_custom.fit(
    X_train_c, y_train_c,
    eval_set=[(X_val_c, y_val_c)],
    verbose=False,
)

pred_custom = model_custom.predict(X_val_c)
mae_custom  = mean_absolute_error(y_val_c, pred_custom)
rmse_custom = mean_squared_error(y_val_c, pred_custom) ** 0.5
r2_custom   = r2_score(y_val_c, pred_custom)

# Spike performance
val_df_c = df_custom_model.loc[val_mask_c].copy()
val_df_c["pred_custom"] = pred_custom
spike_mask_c = val_df_c["da_rt_spread_mwh"].abs() > 20
mae_custom_spike = mean_absolute_error(
    val_df_c.loc[spike_mask_c, TARGET],
    val_df_c.loc[spike_mask_c, "pred_custom"]
)

print("=" * 50)
print(f"{'Metric':<15} {'Baseline':>12} {'Custom':>12} {'Improvement':>12}")
print("-" * 50)
print(f"{'MAE ($/MWh)':<15} {mae_baseline:>12.3f} {mae_custom:>12.3f} {mae_baseline-mae_custom:>+12.3f}")
print(f"{'RMSE ($/MWh)':<15} {rmse_baseline:>12.3f} {rmse_custom:>12.3f} {rmse_baseline-rmse_custom:>+12.3f}")
print(f"{'R²':<15} {r2_baseline:>12.4f} {r2_custom:>12.4f} {r2_custom-r2_baseline:>+12.4f}")
print("=" * 50)
print(f"\nSpike-hour MAE: baseline={mean_absolute_error(val_df.loc[spike_mask,'rt_lbmp_mwh'], val_df.loc[spike_mask,'pred_baseline']):.3f}  custom={mae_custom_spike:.3f}")

In [ ]:
# ── Visualize custom feature importances ─────────────────────────────────────
feat_imp_c = pd.Series(
    model_custom.feature_importances_,
    index=CUSTOM_FEATURES
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
feat_imp_c.head(25).plot(kind="barh", ax=ax, color="steelblue")
ax.invert_yaxis()
ax.set_xlabel("Feature Importance (gain)")
ax.set_title("Top-25 Custom Model Feature Importances")
plt.tight_layout()
plt.savefig("data/plot6_feat_importance_custom.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Section 6: Model Export

Save your model to `submission/ml_model.pkl`. The autograder will load this file.

In [ ]:
import pickle

def save_ml_model(model, feature_fn, feature_cols, mae_val, path="submission/ml_model.pkl"):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    obj = {
        "model": model,
        "feature_fn": feature_fn,
        "feature_cols": feature_cols,
        "target": "rt_lbmp_mwh",
        "metadata": {
            "mae_val": mae_val,
            "model_type": "XGBoost",
            "val_period": "2025-09-01 to 2025-12-31",
        }
    }
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"Saved ML model to {path}  (MAE={mae_val:.3f} $/MWh)")


def load_ml_model(path="submission/ml_model.pkl"):
    with open(path, "rb") as f:
        return pickle.load(f)


save_ml_model(
    model_custom,
    engineer_features_custom,
    CUSTOM_FEATURES,
    mae_custom,
    path="submission/ml_model.pkl"
)

# Quick smoke-test
loaded = load_ml_model("submission/ml_model.pkl")
test_pred = loaded["model"].predict(X_val_c[:5])
print(f"Smoke-test predictions: {test_pred.round(2)}")
print("\nNotebook 1 complete. Proceed to 02_dl_forecasting.ipynb")